# ZarAI — QLoRA Fine-Tuning on Your Own Dataset (Llama-3.1-8B-Instruct)

This notebook fine-tunes **Llama-3.1-8B-Instruct** on your ~1,000-row agricultural
Q&A dataset using **QLoRA** (4-bit quantized base model + LoRA adapters), then
evaluates the fine-tuned model on a **held-out test split** using the same
SBERT semantic accuracy scoring you've used for every other model so far.

Why Llama-3.1-8B as the base: it's your strongest performer across your four
evaluation runs so far (mean 57.4/100), so fine-tuning starts from your best
foundation rather than a weaker one.

**Why a held-out test split matters:** if we evaluated on the same rows the
model was trained on, high scores would just mean it memorized answers —
not that it generalizes. This notebook splits your dataset **before** training
so the final numbers are an honest measure of how well the model performs on
questions it never saw during fine-tuning.

**Before running:**
1. You need Hugging Face access to `meta-llama/Llama-3.1-8B-Instruct` (same as your earlier Llama notebook — you should already have this approved)
2. This will take considerably longer than an evaluation-only run — expect the training step alone to take roughly 30–90+ minutes on a T4, depending on how many epochs and how much of the dataset you use
3. Keep an eye on Colab's session limits; training checkpoints are saved periodically so a disconnect doesn't lose everything

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate "bitsandbytes>=0.46.1" peft trl datasets pandas openpyxl sentence-transformers

## 2. Check GPU

Go to **Runtime → Change runtime type → T4 GPU** before running this.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 2b. Hugging Face login (required — Llama-3.1 is a gated model)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Upload your dataset

Click **Choose Files** and select your dataset Excel file.

In [ ]:
from google.colab import files
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print("Uploaded:", DATA_PATH)

Saving urdu- agriculture Question Asnwer.xlsx to urdu- agriculture Question Asnwer.xlsx
Uploaded: urdu- agriculture Question Asnwer.xlsx


In [ ]:
import pandas as pd

df = pd.read_excel(DATA_PATH)
print("Shape:", df.shape)
df.head(3)

Shape: (1000, 13)


,id,question_urdu,question_roman_urdu,question_english,answer_urdu,answer_roman_urdu,answer_english,category,crop,season,region,difficulty_level,source_type
0,1,گندم میں پیلا زنگ کی بیماری کیسے پہچانی جائے؟,Gandum mein peela zang ki bimari kaise pehchan...,How to identify yellow rust disease in wheat?,گندم میں پیلا زنگ کی علامات: پتوں پر زرد دھاری...,Gandum mein peela zang ki alamaat: patton par ...,Yellow rust in wheat shows yellow stripes on l...,disease_management,wheat,rabi,Punjab,intermediate,expert_knowledge
1,2,گندم کو پانی کب دیں؟,Gandum ko paani kab dein?,When to irrigate wheat?,گندم کو 4 سنچائی چاہیں: کراؤن روٹ (21 دن)، کشت...,Gandum ko 4 senchai chahiyein: crown root (21 ...,Wheat needs 4 irrigations: crown root (21 days...,irrigation_management,wheat,rabi,Punjab,beginner,agricultural_extension
2,3,گندم کی بہترین اقسام کون سی ہیں؟,Gandum ki behtareen aqsaam konsi hain?,What are the best wheat varieties?,پنجاب میں NARC-2009، فیصل آباد-2008، اور پاکست...,Punjab mein NARC-2009; Faisalabad-2008; aur Pa...,In Punjab; NARC-2009; Faisalabad-2008; and Pak...,variety_selection,wheat,rabi,Punjab,beginner,agricultural_extension


## 4. Train / test split (held out BEFORE any training happens)

A stratified 85/15 split by `category`, so both sets have a representative
mix of question types. Uses `groupby().sample()` — not
`groupby().apply(lambda x: x.sample(...))`, since the `.apply()` version
silently drops the grouping column under pandas 3.0+.

`test_df` is set aside now and never touched during training — it's only
used in the final evaluation section.

In [ ]:
TEST_FRACTION = 0.15
RANDOM_STATE = 42

test_df = (
    df.groupby("category", group_keys=False)
      .sample(frac=TEST_FRACTION, random_state=RANDOM_STATE)
)
train_df = df.drop(test_df.index).reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train rows: {len(train_df)}  |  Test rows (held out): {len(test_df)}")
print("\nTrain category distribution:")
print(train_df["category"].value_counts(normalize=True).round(2))
print("\nTest category distribution:")
print(test_df["category"].value_counts(normalize=True).round(2))

Train rows: 851  |  Test rows (held out): 149

Train category distribution:
category
pest_management               0.15
disease_management            0.14
nutrient_deficiency           0.07
irrigation_management         0.07
post_harvest                  0.06
fertilizer_management         0.05
weed_management               0.05
organic_farming               0.04
modern_technology             0.04
yield_improvement             0.03
crop_establishment            0.03
climate_adaptation            0.02
soil_management               0.02
seed_treatment                0.02
planting_calendar             0.02
water_management              0.02
variety_selection             0.02
organic_pest_management       0.01
soil_fertility                0.01
crop_physiology               0.01
integrated_pest_management    0.01
agricultural_challenges       0.01
nursery_management            0.01
soil_health                   0.01
land_preparation              0.01
water_stress                  0.01
crop_

## 5. Build training examples

One training example per (row, language) pair — so a single dataset row with
all three languages filled in produces up to 3 training examples. This
teaches the model all three languages from the same underlying training set,
consistent with how you've been evaluating it.

In [ ]:
SYSTEM_PROMPT = "You are a helpful agricultural advisor for farmers in Pakistan. Answer clearly and practically."

lang_cols = {
    "urdu": ("question_urdu", "answer_urdu"),
    "roman_urdu": ("question_roman_urdu", "answer_roman_urdu"),
    "english": ("question_english", "answer_english"),
}

def build_training_examples(source_df):
    examples = []
    for _, row in source_df.iterrows():
        for lang, (qcol, acol) in lang_cols.items():
            q, a = row.get(qcol), row.get(acol)
            if pd.notna(q) and pd.notna(a) and str(q).strip() and str(a).strip():
                examples.append({
                    "language": lang,
                    "question": str(q).strip(),
                    "answer": str(a).strip(),
                })
    return examples

train_examples = build_training_examples(train_df)
print(f"Total training examples (row x language pairs): {len(train_examples)}")

Total training examples (row x language pairs): 2553


## 6. Load Llama-3.1-8B-Instruct in 4-bit (base for QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 256  # keeps sequences short to help fit training on a T4's 16GB

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,  # bf16 throughout avoids GradScaler/fp16 dtype conflicts entirely
    device_map="auto"
)
print("Base model loaded (4-bit):", MODEL_NAME)
print("GPU memory allocated: {:.2f} GB".format(torch.cuda.memory_allocated() / 1e9))

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Base model loaded (4-bit): meta-llama/Llama-3.1-8B-Instruct
GPU memory allocated: 5.71 GB


## 6b. Check for a previously trained adapter (resume instead of retrain)

If you already trained and downloaded an adapter zip in an earlier session,
upload it here and this cell will extract and load it automatically —
skipping training entirely.

If you don't have one yet, just click **Cancel** on the upload dialog and
the notebook will fall through to Sections 7–9 and train a new adapter
from scratch, same as before.

In [ ]:
import os, zipfile
from google.colab import files

ADAPTER_DIR = "zarai_llama31_qlora_adapter"

print("If you have a previously trained adapter zip, upload it now.")
print("If you don't have one yet, click Cancel to skip and train a new adapter instead.")
uploaded_adapter = files.upload()

if uploaded_adapter:
    adapter_zip_path = list(uploaded_adapter.keys())[0]
    print(f"Found uploaded adapter zip: {adapter_zip_path}")
    os.makedirs(ADAPTER_DIR, exist_ok=True)
    with zipfile.ZipFile(adapter_zip_path, "r") as zf:
        zf.extractall(ADAPTER_DIR)
    print("Extracted to:", ADAPTER_DIR)
    print("Contents:", os.listdir(ADAPTER_DIR))
    ALREADY_TRAINED = True
else:
    print("No adapter uploaded — will train a new one from scratch below (Sections 7-9).")
    ALREADY_TRAINED = False

If you have a previously trained adapter zip, upload it now.
If you don't have one yet, click Cancel to skip and train a new adapter instead.


Saving zarai_llama31_qlora_adapter.zip to zarai_llama31_qlora_adapter.zip
Found uploaded adapter zip: zarai_llama31_qlora_adapter.zip
Extracted to: zarai_llama31_qlora_adapter
Contents: ['tokenizer_config.json', 'training_args.bin', 'checkpoint-640', 'checkpoint-960', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'chat_template.jinja', 'README.md']


## 6c. Load the adapter (only runs if an existing adapter was uploaded above)

In [ ]:
from peft import PeftModel

if ALREADY_TRAINED:
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    model.eval()
    print("Loaded previously trained adapter into model — skipping Sections 7-9 (training).")
else:
    print("Skipping this cell — no existing adapter found. Proceed to Sections 7-9 to train one.")

Loaded previously trained adapter into model — skipping Sections 7-9 (training).


## 7. Format training examples using the chat template

In [ ]:
if not ALREADY_TRAINED:
    from datasets import Dataset

    def format_example(ex):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": ex["question"]},
            {"role": "assistant", "content": ex["answer"]},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return {"text": text}

    formatted = [format_example(ex) for ex in train_examples]
    train_dataset = Dataset.from_list(formatted)
    print(train_dataset)
    print("\n--- Example formatted training text ---\n")
    print(train_dataset[0]["text"])

## 8. Attach LoRA adapters (QLoRA setup)

In [ ]:
if not ALREADY_TRAINED:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    base_model = prepare_model_for_kbit_training(base_model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()

## 9. Train

`NUM_EPOCHS = 3` is a reasonable starting point for ~2,500 examples on an 8B
model — enough to adapt without badly overfitting a dataset this size. Watch
the training loss: if it keeps dropping sharply through all 3 epochs with no
sign of leveling off, more epochs might help; if it flattens early, fewer
would avoid wasted compute.

In [ ]:
if not ALREADY_TRAINED:
    from trl import SFTTrainer, SFTConfig

    OUTPUT_DIR = "zarai_llama31_qlora_adapter"

    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=1,      # small batch to fit T4 memory
        gradient_accumulation_steps=8,       # keeps effective batch size at 8
        gradient_checkpointing=True,         # trades some speed for a lot of memory savings
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=2,
        bf16=True,                           # bf16 needs no loss scaler, avoiding fp16/bf16 dtype conflicts
        optim="paged_adamw_8bit",
        report_to="none",
        dataset_text_field="text",
        loss_type="nll",  # falls back from chunked_nll, which errors on PEFT + gradient checkpointing setups (AttributeError: functools.partial has no __func__)
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )

    trainer.train()

In [ ]:
if not ALREADY_TRAINED:
    # Save the LoRA adapter (small — just the fine-tuned deltas, not the full 8B model)
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Adapter saved to:", OUTPUT_DIR)

    # Zip and download so you don't lose it if the Colab session ends
    import shutil
    shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
    files.download(f"{OUTPUT_DIR}.zip")

## 10. Define the inference function

Works the same way whether `model` came from Section 6c (resumed adapter) or Section 9 (freshly trained) — no changes needed here either way.

In [ ]:
model.eval()

def ask_model(question, system_prompt=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.3,
            do_sample=True
        )
    generated = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# quick sanity check
print(ask_model("گندم کو پانی کب دیں؟"))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


گندم کو 4 سنچائی چاہیں: کھاد کے بعد، پھول آنے سے پہلے، دانہ بھرتے وقت، اور دانہ پکنے کے آخر میں۔


## 11. Load the SBERT scoring model (identical to your other notebooks)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print("Embedding model loaded:", EMBED_MODEL_NAME)

def semantic_similarity(text_a, text_b):
    emb = embed_model.encode([str(text_a), str(text_b)], normalize_embeddings=True)
    return float(np.dot(emb[0], emb[1])) * 100

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


## 12. Evaluate on the held-out test split (honest generalization score)

This is the critical difference from a normal evaluation run: **only
`test_df` is used here** — rows the fine-tuned model never saw during
training. This is what actually tells you whether fine-tuning improved
real performance, not just memorization.

Same SBERT scoring logic as every other model you've evaluated, so this
result is directly comparable to your Qwen/Llama/Qalb numbers.

In [ ]:
import time
from tqdm.auto import tqdm

LANGUAGES_TO_RUN = ["urdu", "roman_urdu", "english"]
CHECKPOINT_EVERY = 25
CHECKPOINT_PATH = "zarai_qlora_test_validation_checkpoint.xlsx"

full_results = []
start_time = time.time()
processed = 0

for lang in LANGUAGES_TO_RUN:
    qcol, acol = lang_cols[lang]
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"Evaluating ({lang})"):
        question = row.get(qcol)
        reference_answer = row.get(acol)

        if pd.isna(question) or not str(question).strip():
            continue

        model_answer = ask_model(str(question))

        if pd.notna(reference_answer) and str(reference_answer).strip():
            accuracy_score = semantic_similarity(model_answer, reference_answer)
        else:
            accuracy_score = None

        full_results.append({
            "id": row.get("id"),
            "language": lang,
            "question": question,
            "model_answer": model_answer,
            "reference_answer": reference_answer,
            "semantic_accuracy_score": accuracy_score,
            "category": row.get("category"),
            "crop": row.get("crop"),
            "season": row.get("season"),
            "region": row.get("region"),
            "difficulty_level": row.get("difficulty_level"),
            "source_type": row.get("source_type"),
        })

        processed += 1
        if processed % CHECKPOINT_EVERY == 0:
            pd.DataFrame(full_results).to_excel(CHECKPOINT_PATH, index=False)

elapsed = time.time() - start_time
print(f"Done. Evaluated {len(full_results)} held-out question/answer pairs in {elapsed/60:.1f} minutes.")

results_df = pd.DataFrame(full_results)
results_df.head()

Evaluating (urdu):   0%|          | 0/149 [00:00<?, ?it/s]

Evaluating (roman_urdu):   0%|          | 0/149 [00:00<?, ?it/s]

Evaluating (english):   0%|          | 0/149 [00:00<?, ?it/s]

Done. Evaluated 447 held-out question/answer pairs in 34.3 minutes.


,id,language,question,model_answer,reference_answer,semantic_accuracy_score,category,crop,season,region,difficulty_level,source_type
0,986,urdu,زراعت میں بہتری کے لیے نکتہ 7 کیا ہے؟,زراعت میں بہتری کے لیے نئی تکنیکیں سیکھیں اور ...,زراعت میں بہتری کے لیے نئی تکنیکیں سیکھیں اور ...,99.999988,agricultural_challenges,rice,all_seasons,All Pakistan,intermediate,agricultural_extension
1,130,urdu,پاکستان میں زراعت کے لیے حکومت کی سبسڈی کیا ہیں؟,پاکستان میں زراعت کے لیے کسان کریڈٹ کارڈ، بیج ...,کپاس، گندم، ڈرپ آبپاشی، سولر پمپ، اور بیج پر س...,69.071096,agricultural_finance,general,all_seasons,All Pakistan,intermediate,agricultural_extension
2,953,urdu,گرم ہوا میں فصل کو کیسے بچائیں؟,گرم ہوا میں ملچ کا استعمال استعمال کریں۔,گرم ہوا میں سوخت مزاحم اقسام استعمال کریں۔,66.451889,climate_adaptation,general,all_seasons,All Pakistan,intermediate,expert_knowledge
3,562,urdu,چاول میں AWD طریقہ سے میتھین گیس کم کرنے کا فا...,ماحول دوست اور گلوبل وارمنگ میں کمی آتی ہے۔,ماحول دوست اور گلوبل وارمنگ میں کمی آتی ہے۔,100.000000,climate_adaptation,rice,kharif,Punjab/Sindh,intermediate,expert_knowledge
4,481,urdu,مکئی میں دانہ بھرنے کے مرحلے میں گرمی سے بچاؤ؟,2% یوریا + پوٹاشیم نائٹریٹ foliar اسپرے کریں۔,2% یوریا foliar اسپرے کریں اور پانی کا خیال رک...,83.873034,climate_adaptation,maize,kharif,All Pakistan,intermediate,expert_knowledge


## 13. Aggregate stats + export full report

In [ ]:
def summarize(group_col):
    valid = results_df.dropna(subset=["semantic_accuracy_score"])
    return (
        valid.groupby(group_col)["semantic_accuracy_score"]
             .agg(["count", "mean", "median", "std", "min", "max"])
             .round(1)
             .sort_values("mean", ascending=False)
    )

overall = pd.DataFrame({
    "metric": ["total_evaluated", "scored_pairs", "mean_score", "median_score", "std_score"],
    "value": [
        len(results_df),
        results_df["semantic_accuracy_score"].notna().sum(),
        round(results_df["semantic_accuracy_score"].mean(), 1),
        round(results_df["semantic_accuracy_score"].median(), 1),
        round(results_df["semantic_accuracy_score"].std(), 1),
    ],
})

by_language = summarize("language")
by_category = summarize("category")
by_crop = summarize("crop")
by_difficulty = summarize("difficulty_level")

low_accuracy_flags = results_df[results_df["semantic_accuracy_score"] < 40]

print("Overall (held-out test set only):")
print(overall.to_string(index=False))
print("\nBy language:")
print(by_language)
print(f"\nRows with low semantic accuracy (<40/100): {len(low_accuracy_flags)}")

Overall (held-out test set only):
         metric  value
total_evaluated  447.0
   scored_pairs  447.0
     mean_score   80.6
   median_score   81.8
      std_score   15.7

By language:
            count  mean  median   std   min    max
language                                          
english       149  84.0    85.4  12.9  43.5  100.0
roman_urdu    149  80.4    84.0  17.0  35.8  100.0
urdu          149  77.5    79.2  16.4  28.8  100.0

Rows with low semantic accuracy (<40/100): 5


In [ ]:
FULL_REPORT_PATH = "zarai_qlora_llama31_test_validation.xlsx"

with pd.ExcelWriter(FULL_REPORT_PATH, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="test_set_results", index=False)
    overall.to_excel(writer, sheet_name="overall_summary", index=False)
    by_language.to_excel(writer, sheet_name="by_language")
    by_category.to_excel(writer, sheet_name="by_category")
    by_crop.to_excel(writer, sheet_name="by_crop")
    by_difficulty.to_excel(writer, sheet_name="by_difficulty")
    low_accuracy_flags.to_excel(writer, sheet_name="flag_low_accuracy", index=False)

files.download(FULL_REPORT_PATH)
print("Exported:", FULL_REPORT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Exported: zarai_qlora_llama31_test_validation.xlsx
